<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 7 · Visualización avanzada y tableros

La semana pasada hiciste figuras que se explican solas. Hoy das el salto que separa un cuaderno de un
producto: una sola pantalla que un gerente abre, entiende y usa para decidir, sin que tú estés al lado.
El cambio no es técnico —Plotly se aprende en veinte minutos— sino de método: **primero se escribe la
pregunta gerencial, después se elige qué figuras la contestan, y todo lo demás se borra**. Al final del
cuaderno vas a haber contestado una pregunta real de Comercial Andina con cuatro figuras, y vas a poder
demostrar por qué las otras dieciséis que se te ocurrieron no entraron.

> **Hoy haces** · Tus primeras figuras interactivas con Plotly y la regla para decidir cuándo la
> interactividad ayuda y cuándo es una excusa para no elegir el mensaje (90 min). Construyes el estado
> de resultados de Comercial Andina desde las ventas y lo cuentas en una cascada anotada, legible por
> alguien sin formación financiera. Compones un tablero de cuatro figuras que contesta **una** pregunta
> gerencial y escribes la recomendación que se deriva. Dejas Streamlit apuntado como el paso siguiente.
>
> **Entrega** · Este cuaderno ejecutado, la cascada del estado de resultados con sus anotaciones, el
> tablero de cuatro paneles con la pregunta escrita arriba y la recomendación abajo, y la lista de las
> figuras que descartaste con el motivo de cada descarte.
> Nombre de archivo: `lab_07_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
BASE_URL = ("https://raw.githubusercontent.com/mayait/"
            "CursoAnalisisDatos_IA_2026/main/sitio/datos")
ARCHIVOS = ["clientes.csv", "productos.csv", "sucursales.csv", "ventas.csv",
            "ventas_limpias.csv", "marketing_mensual.csv",
            "experimento_reactivacion.csv"]
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if (p / "ventas.csv").exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se descargan los siete archivos una vez.
    from urllib.request import urlretrieve
    DATOS = Path("datos")
    DATOS.mkdir(exist_ok=True)
    for archivo in ARCHIVOS:
        if not (DATOS / archivo).exists():
            urlretrieve(f"{BASE_URL}/{archivo}", DATOS / archivo)

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Plotly en el cuaderno

Plotly dibuja en el navegador: la figura que produce es HTML con datos dentro, no una imagen. Eso trae
tres cosas gratis —**hover con el dato exacto, zoom y leyenda que enciende y apaga series**— y trae un
costo: la figura no se puede pegar en un PDF ni imprimir. Las dos bibliotecas conviven. Matplotlib para
lo que se imprime; Plotly para lo que se explora y para lo que se entrega como tablero.

In [ ]:
# En Colab, Plotly ya viene instalado. En tu máquina: pip install plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly
print(f"plotly {plotly.__version__}")

ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")
sucursales = pd.read_csv(DATOS / "sucursales.csv")
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = (ventas
     .assign(mes=lambda d: d["fecha"].dt.to_period("M").dt.to_timestamp())
     .merge(clientes[["cliente_id", "ciudad", "tipo_cliente"]], on="cliente_id",
            how="left", validate="m:1")
     .merge(productos[["producto_id", "nombre", "categoria", "subcategoria", "costo_unitario"]],
            on="producto_id", how="left", validate="m:1")
     .merge(sucursales[["sucursal_id", "canal"]], on="sucursal_id", how="left", validate="m:1"))

v["bruto"] = v["cantidad"] * v["precio_unitario"]          # a precio de lista
v["descuento_valor"] = v["bruto"] * v["descuento"]
v["monto"] = v["bruto"] - v["descuento_valor"]             # lo que se factura
v["costo"] = v["cantidad"] * v["costo_unitario"]
v["margen"] = v["monto"] - v["costo"]

PALETA = {"Online": "#4C72B0", "Tienda": "#B0B0B0"}
print(f"\n{len(v):,} líneas · facturación neta {v['monto'].sum():,.2f} · margen {v['margen'].sum():,.2f}")

## 2. La primera figura interactiva

Misma serie mensual de la semana pasada, ahora abierta por canal. Prueba a pasar el cursor por encima,
a hacer zoom sobre 2025 y a apagar una de las dos series haciendo clic en la leyenda. Esas tres cosas
son todo lo que la interactividad aporta de verdad.

In [ ]:
mensual_canal = (v[v["mes"] < "2026-07-01"]
                 .groupby(["mes", "canal"])["monto"].sum().reset_index())

fig = px.line(mensual_canal, x="mes", y="monto", color="canal", markers=True,
              color_discrete_map=PALETA, template="plotly_white",
              labels={"mes": "", "monto": "facturación neta del mes", "canal": ""},
              title="La tienda física manda todos los meses y el canal en línea no se le acerca")
fig.update_traces(hovertemplate="%{x|%b %Y}<br>%{y:,.0f}<extra>%{fullData.name}</extra>")
fig.update_layout(height=420, hovermode="x unified",
                  legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom"))
fig.add_annotation(x="2025-12-01", y=107942, text="dic-2025: el mejor mes de la tienda",
                   showarrow=True, arrowhead=2, ax=-60, ay=-40, font=dict(color="#C44E52"))
fig.show()

El `hovermode="x unified"` es la única opción de esta figura que cambia algo de verdad: pone los dos
valores del mes en la misma caja, que es como se comparan. El resto —zoom, leyenda interactiva— son
herramientas para **explorar**, no para comunicar. Y ahí está la primera decisión de la semana.

## 3. Qué agrega la interactividad y qué estorba

La interactividad sirve para tres cosas y solo tres:

1. **Detalle bajo demanda** · el número exacto sin ensuciar la figura con etiquetas.
2. **Filtro** · cuando el usuario tiene una pregunta que tú no puedes anticipar (*«¿y en mi sucursal?»*).
3. **Zoom** · cuando la serie es larga y hay que mirar un tramo.

Estorba cuando sustituye a la decisión de qué mostrar. El síntoma es un gráfico que lo dibuja todo y le
pasa al lector el trabajo de encontrar el mensaje.

In [ ]:
por_producto = (v[v["mes"] < "2026-07-01"]
                .groupby(["mes", "nombre"])["monto"].sum().reset_index())

feo = px.line(por_producto, x="mes", y="monto", color="nombre", template="plotly_white",
              title="Facturación mensual por producto (74 series, leyenda incluida)")
feo.update_layout(height=430, showlegend=True)
feo.show()

print(f"series dibujadas : {por_producto['nombre'].nunique()}")
print(f"puntos dibujados : {len(por_producto):,}")
print("preguntas que contesta: 0")

Setenta y cuatro series, 2 220 puntos y ninguna conclusión. La figura es interactiva —se puede aislar
un producto haciendo clic— y eso es justamente el problema: **la interactividad se usó como coartada
para no elegir**. La versión útil elige tres productos y explica por qué esos tres.

In [ ]:
top3 = v.groupby("nombre")["monto"].sum().nlargest(3).index.tolist()
foco = por_producto[por_producto["nombre"].isin(top3)]

bueno = px.line(foco, x="mes", y="monto", color="nombre", template="plotly_white", markers=True,
                labels={"mes": "", "monto": "facturación del mes", "nombre": ""},
                title=(f"Los tres productos más vendidos concentran el "
                       f"{v[v['nombre'].isin(top3)]['monto'].sum() / v['monto'].sum():.1%} "
                       f"de la facturación y se mueven juntos"))
bueno.update_layout(height=400, hovermode="x unified",
                    legend=dict(orientation="h", y=1.02, x=1, xanchor="right", yanchor="bottom"))
bueno.show()

print("Los tres elegidos:", ", ".join(top3))
print(f"series dibujadas: 3 · puntos: {len(foco):,} · preguntas que contesta: 1")

📌 La regla operativa: **antes de añadir un filtro, escribe la pregunta que ese filtro contesta**. Si no
puedes escribirla, el filtro está ahí para que decida el lector, y el lector no va a decidir: va a
cerrar el tablero.

## 4. La cascada: contar un estado de resultados

Un estado de resultados es una resta larga: se parte del ingreso y se van quitando cosas hasta llegar
al margen. Escrito en una tabla, nadie sin formación financiera lo sigue. Dibujado como cascada, se
entiende sin explicación: cada barra flotante es una cosa que se resta y las barras apoyadas en el
suelo son los subtotales.

Primero se construye, línea por línea, desde los datos. Nada de números escritos a mano.

In [ ]:
solo_ventas = v[~v["es_devolucion"]]
devoluciones = v[v["es_devolucion"]]

ingreso_lista = solo_ventas["bruto"].sum()
descuentos = -solo_ventas["descuento_valor"].sum()
devuelto = devoluciones["monto"].sum()
ingreso_neto = v["monto"].sum()
costo_producto = -v["costo"].sum()
margen_bruto = v["margen"].sum()
publicidad = -marketing[["inversion_radio", "inversion_digital", "inversion_volantes"]].sum().sum()
margen_final = margen_bruto + publicidad

estado = pd.DataFrame([
    ("Ingreso a precio de lista", ingreso_lista, "ventas · cantidad × precio_lista"),
    ("Descuentos comerciales", descuentos, "ventas · columna descuento"),
    ("Devoluciones", devuelto, "ventas · es_devolucion"),
    ("Ingreso neto", ingreso_neto, "subtotal"),
    ("Costo de producto", costo_producto, "productos · costo_unitario"),
    ("Margen bruto", margen_bruto, "subtotal"),
    ("Inversión publicitaria", publicidad, "marketing_mensual · tres canales"),
    ("Margen después de marketing", margen_final, "subtotal"),
], columns=["concepto", "valor", "de dónde sale"])
estado["% del ingreso neto"] = (estado["valor"] / ingreso_neto * 100).round(2)

cuadra = abs((ingreso_lista + descuentos + devuelto) - ingreso_neto) < 0.01
print(f"¿el estado de resultados cuadra con la facturación de la semana 5? {cuadra}\n")
estado

Ocho líneas y las cuatro fuentes del curso metidas en una sola historia. Fíjate en las dos que casi
nadie mira: **los descuentos comerciales se llevan 63 681,67 y las devoluciones 67 984,88**, o sea el
2,27 % y el 2,42 % del ingreso neto. Juntos son 131 666,55, el 62,9 % de todo el presupuesto
publicitario de los dos años y medio. Nadie los discute nunca porque no aparecen en ningún informe: se
esconden dentro de la palabra «ventas».

Ahora la misma tabla, contada.

In [ ]:
etiquetas = estado["concepto"].tolist()
valores = estado["valor"].tolist()
medidas = ["absolute", "relative", "relative", "total", "relative", "total", "relative", "total"]

cascada = go.Figure(go.Waterfall(
    orientation="v", measure=medidas, x=etiquetas, y=valores,
    text=[f"{val:,.0f}" for val in valores], textposition="outside",
    connector=dict(line=dict(color="#B0B0B0", width=1)),
    increasing=dict(marker_color="#55A868"),
    decreasing=dict(marker_color="#C44E52"),
    totals=dict(marker_color="#4C72B0"),
    hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>"))

cascada.update_layout(
    template="plotly_white", height=520, showlegend=False,
    title=(f"De cada dólar facturado, Comercial Andina se queda "
           f"{margen_final / ingreso_neto:.2f} después de producto y publicidad"),
    yaxis_title="dólares acumulados 2024-2026", xaxis_tickangle=-25,
    margin=dict(t=90, b=140))
cascada.show()

Se lee de izquierda a derecha como una frase: **entra 2 938 317 a precio de lista, se descuenta, se
devuelve, queda 2 806 651 de ingreso neto; el producto cuesta 1 569 130 y deja 1 237 521 de margen
bruto; la publicidad se lleva 209 425 y quedan 1 028 096.** Nadie necesitó saber qué es un estado de
resultados para seguirla.

## 5. Anotar es decidir

Una figura sin anotar deja la conclusión al lector. Anotar es escribir la conclusión encima del dato
que la sostiene, y obliga a elegir cuál es. Dos anotaciones por figura como máximo: la tercera ya es
ruido.

In [ ]:
cascada_anotada = go.Figure(cascada)
cascada_anotada.add_annotation(
    x="Costo de producto", y=ingreso_neto * 0.72,
    text=f"<b>El producto se lleva {abs(costo_producto) / ingreso_neto:.0%}</b><br>de cada dólar que entra",
    showarrow=True, arrowhead=2, ax=70, ay=-70, align="left",
    bgcolor="rgba(255,255,255,0.85)", bordercolor="#C44E52", font=dict(color="#C44E52", size=12))
cascada_anotada.add_annotation(
    x="Inversión publicitaria", y=margen_bruto * 0.55,
    text=(f"<b>Publicidad: {abs(publicidad) / ingreso_neto:.1%} del ingreso</b><br>"
          f"{abs(publicidad):,.0f} en 31 meses"),
    showarrow=True, arrowhead=2, ax=-10, ay=-90, align="left",
    bgcolor="rgba(255,255,255,0.85)", bordercolor="#4C72B0", font=dict(color="#4C72B0", size=12))
cascada_anotada.add_hline(y=margen_final, line_dash="dot", line_color="#55A868",
                          annotation_text=f"margen final {margen_final:,.0f}",
                          annotation_position="top left")
cascada_anotada.update_layout(
    title="El costo de producto explica el 56 % del ingreso: cualquier ahorro empieza por la negociación de compras")
cascada_anotada.show()

El título cambió y con él cambió el mensaje: la primera versión describía el estado de resultados, la
segunda dice dónde hay que ir a buscar el dinero. Mismos datos, misma figura, otra decisión.

## 6. Anatomía de un tablero: la pregunta primero

Un tablero contesta **una** pregunta. Si contesta tres, son tres tableros. La pregunta de hoy la trae
el gerente comercial de Comercial Andina y está escrita antes de abrir el editor:

> **¿Le damos más presupuesto al canal en línea el próximo año?**

Antes de dibujar nada, los números que la contestan. Si la tabla no la contesta, el tablero tampoco.

In [ ]:
resumen_canal = (v.groupby("canal")
    .agg(facturacion=("monto", "sum"), margen=("margen", "sum"),
         facturas=("factura_id", "nunique"), clientes=("cliente_id", "nunique"),
         unidades=("cantidad", "sum"))
    .assign(margen_pct=lambda d: d["margen"] / d["facturacion"] * 100,
            ticket=lambda d: d["facturacion"] / d["facturas"],
            pct_facturacion=lambda d: d["facturacion"] / d["facturacion"].sum() * 100,
            pct_margen=lambda d: d["margen"] / d["margen"].sum() * 100))

mensual = (v[v["mes"] < "2026-07-01"].pivot_table(index="mes", columns="canal",
                                                  values="monto", aggfunc="sum").fillna(0))
mensual["peso_online"] = mensual["Online"] / mensual.sum(axis=1) * 100
peso_2024 = mensual.loc["2024-01-01":"2024-12-01", "peso_online"].mean()
peso_2026 = mensual.loc["2026-01-01":"2026-06-01", "peso_online"].mean()

online = v[v["canal"] == "Online"]
mezcla = online.groupby("tipo_cliente").agg(facturacion=("monto", "sum"),
                                            facturas=("factura_id", "nunique"),
                                            clientes=("cliente_id", "nunique"))
mezcla["% facturación"] = mezcla["facturacion"] / mezcla["facturacion"].sum() * 100
mezcla["% facturas"] = mezcla["facturas"] / mezcla["facturas"].sum() * 100

print(resumen_canal.round(2).to_string())
print(f"\npeso del canal en línea · 2024: {peso_2024:.2f} %  ·  2026 (6 meses): {peso_2026:.2f} %  "
      f"· variación {peso_2026 - peso_2024:+.2f} puntos\n")
print(mezcla.round(2).to_string())

Los cuatro hechos ya están: el canal en línea es el 18,18 % de la facturación y el 18,16 % del margen;
rinde 44,05 % contra 44,10 % de la tienda, o sea **lo mismo**; **no se mueve** —pesaba 18,53 % en 2024
y 18,63 % en el primer semestre de 2026, diez centésimas de punto en dos años y medio—; y el 84,92 % de
su dinero lo ponen mayoristas que son apenas el 20,80 % de sus facturas. Cuatro hechos, cuatro
paneles.

In [ ]:
tablero = make_subplots(
    rows=2, cols=2, vertical_spacing=0.17, horizontal_spacing=0.12,
    subplot_titles=(
        "1 · Pesa lo mismo en facturación que en margen",
        "2 · No crece: dos años y medio planos",
        "3 · El dinero lo ponen los mayoristas",
        "4 · Y su ticket en línea es igual al de la tienda"))

# Panel 1 · cuánto pesa
tablero.add_trace(go.Bar(x=["% facturación", "% margen"],
                         y=[resumen_canal.loc["Online", "pct_facturacion"],
                            resumen_canal.loc["Online", "pct_margen"]],
                         marker_color="#4C72B0",
                         text=[f"{resumen_canal.loc['Online', 'pct_facturacion']:.1f} %",
                               f"{resumen_canal.loc['Online', 'pct_margen']:.1f} %"],
                         textposition="outside", showlegend=False), row=1, col=1)
tablero.update_yaxes(range=[0, 100], title_text="% del total de la red", row=1, col=1)

# Panel 2 · evolución
tablero.add_trace(go.Scatter(x=mensual.index, y=mensual["peso_online"], mode="lines+markers",
                             line=dict(color="#4C72B0", width=2), showlegend=False,
                             hovertemplate="%{x|%b %Y}: %{y:.1f} %<extra></extra>"), row=1, col=2)
tablero.add_hline(y=mensual["peso_online"].mean(), line_dash="dot", line_color="#C44E52",
                  row=1, col=2)
tablero.update_yaxes(range=[0, 35], title_text="% de la facturación del mes", row=1, col=2)

# Panel 3 · quién pone el dinero y quién las facturas
for i, columna in enumerate(["% facturación", "% facturas"]):
    tablero.add_trace(go.Bar(name=columna, x=mezcla.index, y=mezcla[columna],
                             marker_color=["#4C72B0", "#B0B0B0"][i],
                             text=[f"{val:.0f} %" for val in mezcla[columna]],
                             textposition="outside", showlegend=(True)), row=2, col=1)
tablero.update_yaxes(range=[0, 100], title_text="% dentro del canal en línea", row=2, col=1)

# Panel 4 · ticket por canal y tipo de cliente
ticket = (v[~v["es_devolucion"]].groupby(["canal", "tipo_cliente", "factura_id"])["monto"]
          .sum().reset_index())
for tipo, color in [("Mayorista", "#4C72B0"), ("Minorista", "#B0B0B0")]:
    sub = ticket[ticket["tipo_cliente"] == tipo]
    tablero.add_trace(go.Box(x=sub["canal"], y=sub["monto"], name=tipo, marker_color=color,
                             showlegend=False, boxpoints=False), row=2, col=2)
tablero.update_yaxes(title_text="ticket de la factura", row=2, col=2)

tablero.update_layout(
    template="plotly_white", height=760, boxmode="group", barmode="group",
    legend=dict(orientation="h", y=-0.07, x=0.5, xanchor="center"),
    title=dict(text=("<b>¿Le damos más presupuesto al canal en línea?</b><br>"
                     "<sup>No para crecer: el canal lleva dos años y medio plano en el 18 %. Sí para "
                     "servir mejor a los 308 mayoristas que ya ponen el 85 % de sus ingresos.</sup>"),
               x=0.02),
    margin=dict(t=120, b=90))
tablero.show()

**La respuesta, escrita.** El canal en línea de Comercial Andina se comporta como un canal de
autoservicio para mayoristas que ya son clientes, no como un motor de crecimiento. Pesa 18,18 % de la
facturación con el mismo margen que la tienda, no ha ganado ni una décima de punto en dos años y medio,
y el 84,92 % de sus ingresos viene de 308 mayoristas que hacen apenas el 20,80 % de sus facturas.

La recomendación es de dos frases: *no* al presupuesto de crecimiento, porque dos años y medio planos
son evidencia suficiente de que el dinero solo no mueve la aguja; *sí* a invertir en recompra y
reposición automática para esos 308 mayoristas, que es lo único que el canal demuestra saber hacer. Y
la medición, escrita de antemano: si en seis meses el peso del canal no pasa del 20 %, la hipótesis
estaba mal y se revisa.

Fíjate en lo que **no** está en el tablero: el desglose por categoría, el mapa de ciudades, el top de
productos, la comparación con el año anterior mes a mes. Todo eso existe, todo eso es correcto y nada
de eso contesta la pregunta.

## 7. Qué se elimina

Fondo blanco, sin sombras, sin bordes, sin logotipos repetidos, sin leyendas que repiten lo que ya dice
el eje, sin rejilla vertical si las barras son verticales. La regla: **si un elemento no codifica un
dato, sobra.**

In [ ]:
base = v.groupby("ciudad")["monto"].sum().sort_values()

recargada = go.Figure(go.Bar(x=base.values, y=base.index, orientation="h",
                             marker=dict(color=base.values, colorscale="Rainbow",
                                         line=dict(color="black", width=2)),
                             name="facturación"))
recargada.update_layout(title="Facturación por ciudad", template="plotly", height=330,
                        showlegend=True, xaxis_title="facturación (USD)", yaxis_title="Ciudad",
                        paper_bgcolor="#EFEFEF")
recargada.show()

limpia = go.Figure(go.Bar(x=base.values, y=base.index, orientation="h",
                          marker_color=["#B0B0B0"] * 4 + ["#4C72B0"],
                          text=[f"{val / 1000:,.0f}k" for val in base.values],
                          textposition="outside",
                          hovertemplate="%{y}: %{x:,.0f}<extra></extra>"))
limpia.update_layout(
    title=f"Quito factura {base['Quito'] / base['Loja']:.1f} veces lo de Loja y decide el resultado de la red",
    template="plotly_white", height=330, showlegend=False,
    xaxis=dict(visible=False), yaxis_title="", margin=dict(l=90, r=40, t=70, b=20))
limpia.show()

print("Elementos eliminados: escala de color sin significado, borde negro, fondo gris, leyenda de una")
print("sola serie, título de eje redundante, eje x completo (el dato ya está escrito en la barra).")
print("Elementos añadidos: uno. El color que señala a Quito, y el título que dice qué pasa.")

## 8. Streamlit: el paso siguiente

Hasta aquí el tablero vive dentro del cuaderno, y eso tiene un techo: para verlo hay que ejecutar el
cuaderno. Streamlit convierte este mismo código en una aplicación web que el gerente abre en el
navegador, sin Python y sin pedirte permiso. **No se usa hoy** —es materia de la semana 16 y del
proyecto final— pero conviene ver lo poco que cambia:

```python
# tablero.py  ·  se ejecuta con:  streamlit run tablero.py
import streamlit as st

st.title("¿Le damos más presupuesto al canal en línea?")
ciudad = st.selectbox("Ciudad", ["Todas"] + sorted(v["ciudad"].unique()))
datos = v if ciudad == "Todas" else v[v["ciudad"] == ciudad]
st.metric("Peso del canal en línea", f"{peso_online(datos):.1f} %")
st.plotly_chart(construir_tablero(datos), use_container_width=True)
```

Son las mismas figuras de Plotly; lo único nuevo son los controles y el `st.plotly_chart`. Y aparece de
inmediato la tentación de la semana: **ese `selectbox` es un filtro, y ya sabes qué hay que escribir
antes de ponerlo.**

Mientras tanto, un tablero de Plotly se entrega hoy mismo como un archivo HTML que se abre con doble
clic y funciona sin Python instalado.

In [ ]:
import tempfile

destino = Path(tempfile.gettempdir()) / "tablero_canal_online.html"
tablero.write_html(destino, include_plotlyjs="cdn")

print(f"tablero exportado a {destino}")
print(f"tamaño: {destino.stat().st_size / 1024:,.0f} KB · se abre con doble clic, sin instalar nada")
print("include_plotlyjs='cdn' deja el archivo ligero pero exige internet al abrirlo;")
print("con include_plotlyjs=True el archivo pesa unos 3 MB y funciona sin conexión.")

### 🌶️ Ejercicio 1 — Guiado

Construye la cascada del estado de resultados **de una sola ciudad** y compárala con la de la red
completa. Añade dos anotaciones: una sobre la línea que más se desvía del promedio de la red y otra con
la conclusión. Después contesta en texto: ¿el problema de esa ciudad está en el precio, en el descuento,
en la devolución o en el costo?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: filtra v por ciudad y repite el bloque de la sección 4; conviene envolverlo en una función
#          def estado_resultados(datos) -> pd.DataFrame
# Pista 2: para comparar, expresa cada línea como % del ingreso neto de SU ciudad, no en dólares
# Pista 3: la publicidad de marketing_mensual es de toda la red: o la repartes con un criterio
#          explícito o la dejas fuera y lo dices en el título

### 🔥 Desafío · dividido en dos

Cambia la pregunta gerencial y rehaz el tablero: **¿en qué ciudad abrimos la próxima tienda?** Tienes
facturación y margen por ciudad, metros cuadrados de cada tienda en `sucursales.csv`, venta por metro
cuadrado, clientes captados que nunca compraron y peso del canal en línea por ciudad.

**Parte A · en clase (15 min).** La tabla de números que contesta la pregunta —una fila por ciudad, las
columnas que decidas— más **un solo panel**, el que más pese en la recomendación, con la respuesta
escrita en el título. Si la tabla no contesta la pregunta, ninguna figura lo va a arreglar: esa es toda
la lección de la parte A.

**Parte B · para casa (30 min).** El tablero completo de cuatro paneles, ni uno más, con la pregunta
arriba y la recomendación abajo, y una celda de texto con las figuras que descartaste y el motivo de
cada descarte. Va en la entrega del cuaderno.

In [ ]:
# TU CÓDIGO AQUÍ
# --- Parte A (en clase) ---
# Pista 1: empieza por la tabla de números que contesta la pregunta. Si la tabla no la contesta,
#          ninguna figura lo va a arreglar
# Pista 2: la venta por metro cuadrado solo tiene sentido con las ventas de tienda, no con las de línea
#
# --- Parte B (para casa) ---
# Pista 3: make_subplots(rows=2, cols=2, subplot_titles=(...)) y un título general con la respuesta
# Pista 4: el motivo de descarte más frecuente y más válido es «es cierto pero no cambia la decisión»

### 🎯 Reto en clase (15 min)

Presentación cruzada. Cada equipo enseña su tablero durante **tres minutos** sin explicar cómo lo hizo:
solo la pregunta, la respuesta y la evidencia. El resto de equipos anota, antes de que el presentador
diga nada, **qué pregunta cree que contesta el tablero**. Si lo que anotan no coincide con la pregunta
que el equipo escribió, el tablero no comunica y hay que rehacerlo. Después, fuego cruzado: dos
objeciones por equipo, y ninguna puede ser sobre los colores.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: la prueba de los tres segundos. Enseña solo el título y el panel de arriba a la izquierda
# y pregunta qué se decide con eso. Si no hay respuesta, el panel principal está mal elegido.
# tablero.show()  ← empieza por aquí y quita paneles hasta que duela

## La trampa de hoy

⚠️ **Construir el tablero antes de escribir la pregunta que debe contestar.** Se reconoce a simple
vista: tiene siete filtros y ninguna conclusión. El argumento de quien lo hizo siempre es el mismo,
«así cada uno mira lo que necesita», y suena a flexibilidad. Es lo contrario: es trasladarle al gerente
el trabajo de análisis que le tocaba al analista.

Se puede medir. Cuenta cuántas vistas distintas produce un tablero con filtros y cuántas produce el
tablero de la sección 6.

In [ ]:
filtros = {
    "mes": v["mes"].nunique(), "ciudad": v["ciudad"].nunique(),
    "canal": v["canal"].nunique(), "sucursal": v["sucursal_id"].nunique(),
    "tipo de cliente": v["tipo_cliente"].nunique(), "categoría": v["categoria"].nunique(),
    "subcategoría": v["subcategoria"].nunique(),
}

vistas = 1
detalle = []
for nombre, n in filtros.items():
    vistas *= (n + 1)                       # +1 por la opción «Todos»
    detalle.append((nombre, n, n + 1, vistas))
print(pd.DataFrame(detalle, columns=["filtro", "valores", "opciones", "vistas acumuladas"])
      .to_string(index=False))

print(f"\nTABLERO SIN PREGUNTA  · {vistas:,} vistas posibles · 0 conclusiones escritas")
print(f"TABLERO DE LA SECCIÓN 6 · {1:>13,} vista       · 1 conclusión y 1 recomendación")
print(f"\nrazón entre los dos: {vistas:,.0f} a 1")
print(f"a diez segundos por vista, recorrerlas todas cuesta {vistas * 10 / 3600 / 8:,.0f} jornadas de trabajo")

📌 **943 488 vistas posibles contra una.** El tablero con siete filtros no es más flexible: es un
buscador sin buscador, y recorrerlo entero costaría 328 jornadas de trabajo. El tablero de la sección 6
enseña una sola vista, y esa vista trae la respuesta escrita.

Esto no significa que los filtros estén prohibidos. Significa que cada filtro se justifica con la
pregunta que contesta, y que un tablero con siete filtros necesita siete justificaciones escritas. Casi
siempre, al escribirlas, sobran cinco.

La secuencia correcta, y es la que se califica:

1. Escribe la pregunta gerencial en una frase, con signo de interrogación.
2. Escribe la respuesta que esperas y qué decisión cambia según cuál sea.
3. Elige los números que la contestan. Si caben en una tabla, quizá no hacía falta un tablero.
4. Dibuja como máximo cuatro paneles y ponle a cada uno un título que sea una afirmación.
5. Borra todo lo que no participe en la respuesta, aunque te haya costado dos horas.

## Entregable

Sube `lab_07_apellido.ipynb` con:

- La cascada del estado de resultados construida desde los datos —sin un solo número escrito a mano— y
  cuadrada contra la facturación neta de la semana 5: 2 806 650,55.
- La misma cascada anotada, con dos anotaciones como máximo y un título que diga dónde está el dinero.
- El tablero de cuatro paneles con la pregunta gerencial escrita arriba y la recomendación abajo, más
  el archivo HTML exportado.
- La lista de las figuras que descartaste, cada una con el motivo. Se califica esta lista tanto como el
  tablero.
- Una fila en la bitácora de prompts: le describiste tu tablero al asistente sin enseñarle el código y
  le pediste que dijera qué pregunta cree que contesta. Anota qué contestó y qué cambiaste.

## Para tu equipo

- El tablero del caso se entrega esta semana dentro del cuaderno y con **una** pregunta gerencial
  explícita en el título. Un tablero con dos preguntas son dos tableros mal hechos.
- Escriban la pregunta y la respuesta esperada **antes** de abrir el editor, y guarden ese texto: en la
  defensa se compara contra lo que acabó mostrando el tablero.
- Prueben la regla de los tres segundos con alguien ajeno al equipo: si en tres segundos no dice de qué
  va, el panel principal está mal elegido. Es más barato descubrirlo ahora que delante del panel.